# Unit 2: Predictive Modelling & Evaluation

**Course:** 23CSE452 Business Analytics  
**Objective:** Train, tune, and evaluate classification models to identify high-rated movies based on pre-release movie characteristics (content-only) and post-release engagement signals.

### Target Definition:
- **High-rated movie:** `vote_average >= 6.5`
- **Reliable rating threshold:** `vote_count >= 20` (excludes titles with unstable ratings)

### Evaluated Models:
1. **Majority-Class Baseline (DummyClassifier)**
2. **Logistic Regression (L2 Regularized)**
3. **Random Forest Classifier (Ensemble Bagging)**
4. **XGBoost Classifier (Gradient Boosting)**


In [ ]:
import os, ast, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, RocCurveDisplay
)
from sklearn.inspection import permutation_importance
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


## 1. Data Preparation & Feature Sets
- **Set A (Content Only - Pre-release):** Release year, release month, overview length, genre count, multi-label genre indicators, and top 10 language indicators.
- **Set B (Content + Engagement):** Set A + log(1 + popularity) and log(1 + vote_count).


In [ ]:
# Load clean movie dataset
df = pd.read_csv("../data/processed/tmdb_clean.csv", parse_dates=["release_date"])
df["genres"] = df["genres"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Filter to reliable movies (vote_count >= 20)
m = df[df["vote_count"] >= 20].copy().reset_index(drop=True)
m["high_rated"] = (m["vote_average"] >= 6.5).astype(int)
print(f"Modelling movies (>=20 votes): {len(m):,} | High-rated share: {m['high_rated'].mean():.1%}")

# Multi-label genre binarization
mlb = MultiLabelBinarizer()
gx = pd.DataFrame(mlb.fit_transform(m["genres"]), index=m.index,
                  columns=["g_" + c for c in mlb.classes_])

# Top 10 languages + other
top_langs = m["original_language"].value_counts().head(10).index
m["lang"] = m["original_language"].where(m["original_language"].isin(top_langs), "other")
lx = pd.get_dummies(m["lang"], prefix="l", dtype=int)

# Temporal & Text metadata features
m["month"] = m["release_date"].dt.month
m["ov_words"] = m["overview"].fillna("").str.split().str.len()
m["n_genres"] = m["genres"].str.len()
m["log_pop"] = np.log1p(m["popularity"])
m["log_votes"] = np.log1p(m["vote_count"])

# Construct Feature Sets
base_cols = ["release_year", "month", "ov_words", "n_genres"]
X_a = pd.concat([m[base_cols], gx, lx], axis=1)
X_b = pd.concat([X_a, m[["log_pop", "log_votes"]]], axis=1)
y = m["high_rated"]

print(f"Feature Set A (Content Only): {X_a.shape[1]} features")
print(f"Feature Set B (Content + Engagement): {X_b.shape[1]} features")


## 2. Stratified Train / Test Split (80/20)

In [ ]:
tr, te = train_test_split(m.index, test_size=0.2, stratify=y, random_state=42)
y_tr, y_te = y.loc[tr], y.loc[te]

print(f"Training set: {len(tr):,} movies ({y_tr.mean():.1%} high-rated)")
print(f"Test set:     {len(te):,} movies ({y_te.mean():.1%} high-rated)")


## 3. Model Training & Held-Out Test Evaluation
Compare Baseline, Logistic Regression, Random Forest, and XGBoost on both Set A and Set B.


In [ ]:
# Load pre-trained tuned models from models/ directory or evaluate them directly
rf_a = joblib.load("../models/RandForest_A.pkl")
rf_b = joblib.load("../models/RandForest_B.pkl")
xgb_a = joblib.load("../models/XGBoost_A.pkl")
xgb_b = joblib.load("../models/XGBoost_B.pkl")

# Baseline model
dummy = DummyClassifier(strategy="prior")
dummy.fit(X_b.loc[tr], y_tr)

# Evaluate each model on held-out test set
eval_models = [
    ("Majority Baseline", dummy, X_b, False),
    ("Random Forest (Set A: Content Only)", rf_a, X_a, False),
    ("Random Forest (Set B: Content + Engagement)", rf_b, X_b, False),
    ("XGBoost (Set A: Content Only)", xgb_a, X_a, True),
    ("XGBoost (Set B: Content + Engagement)", xgb_b, X_b, True),
]

results = []
for name, mdl, X, is_xgb in eval_models:
    X_test_data = X.loc[te].values if is_xgb else X.loc[te]
    pred = mdl.predict(X_test_data)
    prob = mdl.predict_proba(X_test_data)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, prob)
    })

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))


## 4. Confusion Matrices & ROC Curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix RF Set A
p_rf_a = rf_a.predict(X_a.loc[te])
ConfusionMatrixDisplay(confusion_matrix(y_te, p_rf_a), display_labels=["Low / Medium", "High"]).plot(ax=ax[0], colorbar=False, cmap="Blues")
ax[0].set_title("Confusion Matrix: RF Set A (Content Only)")

# Confusion Matrix RF Set B
p_rf_b = rf_b.predict(X_b.loc[te])
ConfusionMatrixDisplay(confusion_matrix(y_te, p_rf_b), display_labels=["Low / Medium", "High"]).plot(ax=ax[1], colorbar=False, cmap="Blues")
ax[1].set_title("Confusion Matrix: RF Set B (Content + Engagement)")

# ROC Curves
pr_rf_b = rf_b.predict_proba(X_b.loc[te])[:, 1]
pr_rf_a = rf_a.predict_proba(X_a.loc[te])[:, 1]
pr_xgb_b = xgb_b.predict_proba(X_b.loc[te].values)[:, 1]

RocCurveDisplay.from_predictions(y_te, pr_rf_b, ax=ax[2], name="RF Set B (AUC=0.836)", color="#1f77b4")
RocCurveDisplay.from_predictions(y_te, pr_xgb_b, ax=ax[2], name="XGBoost Set B (AUC=0.831)", color="#2ca02c")
RocCurveDisplay.from_predictions(y_te, pr_rf_a, ax=ax[2], name="RF Set A (AUC=0.785)", color="#ff7f0e")
ax[2].plot([0, 1], [0, 1], "k--", label="Chance (AUC=0.500)")
ax[2].legend(loc="lower right")
ax[2].set_title("ROC Curves Comparison")

plt.tight_layout()
plt.show()


## 5. Feature Importance (Permutation Importance on Test Set)

In [ ]:
pi_b = permutation_importance(rf_b, X_b.loc[te], y_te, scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1)
imp_b = pd.Series(pi_b.importances_mean, index=X_b.columns).sort_values().tail(15)

pi_a = permutation_importance(rf_a, X_a.loc[te], y_te, scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1)
imp_a = pd.Series(pi_a.importances_mean, index=X_a.columns).sort_values().tail(15)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
imp_b.plot(kind="barh", ax=ax[0], color="steelblue", title="Top 15 Permutation Importance — Set B (Drop in AUC)")
imp_a.plot(kind="barh", ax=ax[1], color="darkorange", title="Top 15 Permutation Importance — Set A (Drop in AUC)")
ax[0].set_xlabel("Mean Decrease in ROC-AUC")
ax[1].set_xlabel("Mean Decrease in ROC-AUC")
plt.tight_layout()
plt.show()

print("Top 10 features for Set B:")
print(imp_b.sort_values(ascending=False).head(10).round(4).to_string())


## 6. Model Robustness Check (Vote Threshold >= 50)
Verify whether model performance remains consistent when evaluated on movies with higher vote volume.


In [ ]:
r = df[df["vote_count"] >= 50].copy()
r["y"] = (r["vote_average"] >= 6.5).astype(int)
r["log_pop"], r["log_votes"] = np.log1p(r["popularity"]), np.log1p(r["vote_count"])
r["month"], r["ov_words"], r["n_genres"] = r["release_date"].dt.month, r["overview"].fillna("").str.split().str.len(), r["genres"].str.len()
gr_ = pd.DataFrame(MultiLabelBinarizer().fit_transform(r["genres"]), index=r.index)
r["lang"] = r["original_language"].where(r["original_language"].isin(top_langs), "other")
Xr = pd.concat([r[["release_year", "month", "ov_words", "n_genres", "log_pop", "log_votes"]], gr_, pd.get_dummies(r["lang"], dtype=int)], axis=1)
Xr.columns = Xr.columns.astype(str)

a, b, c, d = train_test_split(Xr, r["y"], test_size=0.2, stratify=r["y"], random_state=42)
rf_robust = RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=3, n_jobs=-1, random_state=42).fit(a, c)

print(f"Robustness Dataset (>=50 votes): {len(r):,} movies | Positives: {r['y'].mean():.1%}")
print(f"Test ROC-AUC at >=50 votes: {roc_auc_score(d, rf_robust.predict_proba(b)[:, 1]):.3f}")
